# Before you get started

### You need the following data:

**1) Reference proteomes for each species**

This model works right out of the box with refseq and ensemble proteome header formats from the following sources:
https://www.ncbi.nlm.nih.gov/datasets/
http://ftp.ensembl.org/pub/

if you are using data from one of these sources, download both the .faa and .gtf files. If not, the protomes input into the tool must be in the following format, with gene names that directly match those in the scRNA-seq matrices:

>\>gene_name_1
Sequence
>\>gene_name_2
Sequence

**2) Cleaned scRNA-seq matrices**

Counts should be stored within anndata objects compressed in the h5ad format. Raw counts should be stored in the adata.X slot. Each species should be stored in its own h5ad file. These datasets should be filtered for high quality genes/cells beforehand.

**3) Putative cell type annotations**
CSHint requires that each species have at least 3 cell types in common with the reference species. Cell type annotations do not need to be complete or highly granular. General groupings like "epithelial", "endothelial", "stromal", and "immune" will suffice. Cell type names need to be standardised across species. They should be stored in a single column in the adata.obs slot in the h5ads for each species.

### Dependencies
This tool runs on python 3.10. See the requirements.txt file in this directory for necessary dependencies.

# Preprocessing module

The preprocessing module is an optional module that automates some of the necessary preprocessing for CSHint. Note that these cleaning steps must be done manually if you choose note to use this module, or if your data does not follow standard formats.

### Step 1: clean the proteomes:

In the below step, we are cleaning the proteome headers so that they only contain gene names that match the scRNA-seq matrices. In this case, we are using proteome from both refseq and ensembl, to match the original assemblies used by the authors. The preprocess module can automatically detect these file headers, and parse them accordingly. It will write the cleaned fastas into a new directory

In [1]:
from CSHint.utils.CSHint_preprocess import CSHint_preprocess

pre = CSHint_preprocess()

path_dict = {
    "chicken" : {"gtf" : "../PaperScripts/Spermatogenesis/Data/gtfs/chicken.gtf",
                 "fasta": "../PaperScripts/Spermatogenesis/Data/raw_proteomes/chicken.faa"},
    "human" : {"gtf" : "../PaperScripts/Spermatogenesis/Data/gtfs/human.gtf",
               "fasta": "../PaperScripts/Spermatogenesis/Data/raw_proteomes/human.faa"},
     "mouse" : {"gtf" : "../PaperScripts/Spermatogenesis/Data/gtfs/mouse.gtf",
                "fasta": "../PaperScripts/Spermatogenesis/Data/raw_proteomes/mouse.faa"},
     "Monodelphis" : {"gtf" : "../PaperScripts/Spermatogenesis/Data/gtfs/Monodelphis.gtf",
                      "fasta": "../PaperScripts/Spermatogenesis/Data/raw_proteomes/Monodelphis.faa"},
     "platypus" : {"gtf" : "../PaperScripts/Spermatogenesis/Data/gtfs/platypus.gtf",
                   "fasta": "../PaperScripts/Spermatogenesis/Data/raw_proteomes/platypus.faa"},
     "rhesus" : {"gtf" : "../PaperScripts/Spermatogenesis/Data/gtfs/rhesus.gff",
                 "fasta": "../PaperScripts/Spermatogenesis/Data/raw_proteomes/rhesus.faa"},
     "sheep" : {"gtf" : "../PaperScripts/Spermatogenesis/Data/gtfs/sheep.gtf",
                "fasta": "../PaperScripts/Spermatogenesis/Data/raw_proteomes/sheep.faa"}
}


pre.preprocess_fasta_headers(path_dict, out_dir ="../../CSHint/filtered_proteomes")

[chicken] Converting FASTA headers... (detected format: ensembl)
  [chicken] GTF index (Ensembl): 93,251 accession entries.
  [chicken] Autodetected link field 'ENSGAL' (ENSGAL:200) from 200 headers.
  [chicken] 18308 unique symbols written (0 unmatched).
  [chicken] → filtered_proteomes/chicken.faa
  [chicken] → filtered_proteomes/chicken_id_map.csv
[human] Converting FASTA headers... (detected format: refseq)
  [human] GTF index (RefSeq): 212,424 accession entries.
  [human] Autodetected link field 'NP' (NP:200) from 200 headers.
  [human] 19869 unique symbols written (0 unmatched).
  [human] → filtered_proteomes/human.faa
  [human] → filtered_proteomes/human_id_map.csv
[mouse] Converting FASTA headers... (detected format: refseq)
  [mouse] GTF index (RefSeq): 101,433 accession entries.
  [mouse] Autodetected link field 'NP' (NP:200) from 200 headers.
  [mouse] 21847 unique symbols written (0 unmatched).
  [mouse] → filtered_proteomes/mouse.faa
  [mouse] → filtered_proteomes/mouse_id

### Step 2: Synchronising the protomes and the scRNA-seq.

The following function filters the protome and the scRNA data so that only genes shared between the two are kept for each species. This technically isn't required, however it is HIGHLY recommended to do this to cut down on run times down the line.

In [2]:
path_dict = {
    "chicken" : {"h5ad" : "../PaperScripts/Spermatogenesis/Data/raw_adatas/chicken.h5ad",
                 "fasta": "filtered_proteomes/chicken.faa"},
    "human" : {"h5ad" : "../PaperScripts/Spermatogenesis/Data/raw_adatas/human.h5ad",
               "fasta": "filtered_proteomes/human.faa"},
     "mouse" : {"h5ad" : "../PaperScripts/Spermatogenesis/Data/raw_adatas/mouse.h5ad",
                "fasta": "filtered_proteomes/mouse.faa"},
     "Monodelphis" : {"h5ad" : "../PaperScripts/Spermatogenesis/Data/raw_adatas/Monodelphis.h5ad",
                      "fasta": "filtered_proteomes/Monodelphis.faa"},
     "platypus" : {"h5ad" : "../PaperScripts/Spermatogenesis/Data/raw_adatas/platypus.h5ad",
                   "fasta": "filtered_proteomes/platypus.faa"},
     "rhesus" : {"h5ad" : "../PaperScripts/Spermatogenesis/Data/raw_adatas/rhesus.h5ad",
                 "fasta": "filtered_proteomes/rhesus.faa"},
     "sheep" : {"h5ad" : "../PaperScripts/Spermatogenesis/Data/raw_adatas/sheep.h5ad",
                "fasta": "filtered_proteomes/sheep.faa"}
}

pre.synchronize_scrna_and_proteomes(species_map = path_dict, prot_out_dir="../../CSHint/filtered_proteomes/", scrna_out_dir="../../CSHint/filtered_scrna/")

[chicken] Harmonizing scRNA-seq and Proteome...
  [chicken] Shared genes found: 9207
  [chicken] Saved filtered h5ad to filtered_scrna/chicken.h5ad
  [chicken] Saved filtered proteome to filtered_proteomes/chicken.faa
[human] Harmonizing scRNA-seq and Proteome...
  [human] Shared genes found: 17454
  [human] Saved filtered h5ad to filtered_scrna/human.h5ad
  [human] Saved filtered proteome to filtered_proteomes/human.faa
[mouse] Harmonizing scRNA-seq and Proteome...
  [mouse] Shared genes found: 18629
  [mouse] Saved filtered h5ad to filtered_scrna/mouse.h5ad
  [mouse] Saved filtered proteome to filtered_proteomes/mouse.faa
[Monodelphis] Harmonizing scRNA-seq and Proteome...
  [Monodelphis] Shared genes found: 10498
  [Monodelphis] Saved filtered h5ad to filtered_scrna/Monodelphis.h5ad
  [Monodelphis] Saved filtered proteome to filtered_proteomes/Monodelphis.faa
[platypus] Harmonizing scRNA-seq and Proteome...
  [platypus] Shared genes found: 8938
  [platypus] Saved filtered h5ad to fi

# Running Orthofinder
CSHint requires gene family information. It does this by using orthogroup information from Orthofinder (https://doi.org/10.1186/s13059-019-1832-y). This module comes with a python wrapper for it:

In [3]:
from CSHint.utils.OrthofinderWrapper import OrthoFinderWrapper
ow = OrthoFinderWrapper()
ow.run_orthofinder(inputdir="filtered_proteomes/",
    orthofinder_path="orthofinder",
    threads=16)

Running: orthofinder -f filtered_proteomes/ -t 16

OrthoFinder version 3.0.1b1 Copyright (C) 2014 David Emms

2026-08-31 02:52:32 : Starting OrthoFinder 3.0.1b1
16 thread(s) for highly parallel tasks (BLAST searches etc.)
2 thread(s) for OrthoFinder algorithm

Results directory: /storage/shared/alicia/analysis/CSHInt_github/CSHint/filtered_proteomes/OrthoFinder/Results_Aug31_1/

Checking required programs are installed
----------------------------------------
Running with the recommended MSA tree inference by default. To revert to legacy method use '-M dendroblast'.

Test can run "mcl -h" - ok
Test can run "mafft" - ok
Test can run "fasttree" - ok

Monodelphis_id_map.csv
chicken_id_map.csv
human_id_map.csv
mouse_id_map.csv
platypus_id_map.csv
rhesus_id_map.csv
sheep_id_map.csv
OrthoFinder expects FASTA files to have one of the following extensions: fasta, fas, faa, pep, fa

Dividing up work for BLAST for parallel processing
--------------------------------------------------
2026-08-31 

# Running the count merge module
The countmerge module merges counts together by creating 1-1 pairings of genes across species based on both sequence and expression level orthology using a greedy algorithm. TODO: EXPAND ON HOW THIS WORKS HERE

In [4]:
from CSHint.utils.CSHint_merger import CSHint_merger

import warnings
warnings.filterwarnings("ignore")



scrnadirs = {
    "human": "filtered_scrna/human.h5ad",
    "mouse": "filtered_scrna/mouse.h5ad",
    "rhesus": "filtered_scrna/rhesus.h5ad",
    "sheep" : "filtered_scrna/sheep.h5ad",
    "Monodelphis": "filtered_scrna/Monodelphis.h5ad",
    "platypus": "filtered_scrna/platypus.h5ad",
    "chicken": "filtered_scrna/chicken.h5ad"

}

merger = CSHint_merger(
    orthofinder_resultdir="filtered_proteomes/OrthoFinder/Results_Aug31/",
    scrna_dirs=scrnadirs,
    output_dir="../../CSHint/merged_output/",
    annotation_obs_col = "metacluster", #include the name of the column you put the cell type annotations in here

)

merger.run()

Loading OrthoFinder results...
Loading scRNA-seq data...
  Loading human...
  Loading mouse...
  Loading rhesus...
  Loading sheep...
  Loading Monodelphis...
  Loading platypus...
  Loading chicken...
  Found 5 anchor cell types across all species
Computing anchor cell type means (normalised)...
Computing greedy functional homolog layering...
Filling final matrix...
  24394 rows x 38825 cells; 95.7% of total counts retained in orthogroup layers
Normalising (scale_factor=10000, log1p=True, by=matrix)...
Saving to merged_output/...
Saved 24394 rows and 38825 cells.
Metadata generated: metadata.txt, rowmetadata.csv, normalisation.json
Matrices: matrix.mtx (raw counts), matrix_normalised.mtx (scaled)


# Running the integration module

Now that we have the merged matrix, we can begin the integration steps. First, we need to create a select set of features for integration (this is comparable to running sc.pp.highly_variable_genes() in scanpy or FindVariableFeatures in seurat). For feature selection, it is best to prioritise genes that will have similar expression patterns across cell types and  preserve enough variance to distinguish cell types. Do do this, CSHint uses a two part filter. The discriminability percentile is a measure of a features variance (for measuring it's capacity for cell type separation) and the conservation threshold measures how similar a features expression is across species. These two thresholds can be manually adjusted. It is recommended to leave around 1000 features for integration.

In [1]:
from CSHint.utils.CSHint_integration import CSHint_integration
analysis = CSHint_integration("../../CSHint/merged_output/", reference_species ="human")
analysis.load()

analysis.select_features(
    discriminability_threshold=0.4,
    conservation_threshold=0.5,
)



/home/apetrany/miniconda3/envs/CSHint/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading data from merged_output/...
Calculating cell type means from matrix...


/storage/shared/alicia/analysis/CSHInt_github/CSHint/utils/CSHint_integration.py:198: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  c, _ = spearmanr(vecs[i], vecs[j])


Selected 1760 features.


Next, we can prepare the selected features for integration. This step normalises the data and notes equivalencies across cell types.

In [2]:
adata = analysis.prepare_for_integration()

/home/apetrany/miniconda3/envs/CSHint/lib/python3.10/functools.py:889: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


  Reference species: human (7 cell types)
  Full anchors (all species): 5
  Partial anchors: 2
  Unique cell types: 0
  human <-> mouse: 7 shared cell types (Early Sperm, Early Spermatid, Early Spermatocyte, Late Sperm, Late Spermatid...)
  human <-> rhesus: 5 shared cell types (Early Spermatid, Early Spermatocyte, Late Spermatid, Late Spermatocyte, SSC)
  human <-> sheep: 7 shared cell types (Early Sperm, Early Spermatid, Early Spermatocyte, Late Sperm, Late Spermatid...)
  human <-> Monodelphis: 6 shared cell types (Early Spermatid, Early Spermatocyte, Late Sperm, Late Spermatid, Late Spermatocyte...)
  human <-> platypus: 5 shared cell types (Early Spermatid, Early Spermatocyte, Late Spermatid, Late Spermatocyte, SSC)
  human <-> chicken: 6 shared cell types (Early Spermatid, Early Spermatocyte, Late Sperm, Late Spermatid, Late Spermatocyte...)


Now we can run a brief optimisation protocol to identify the optimal number of anchors for integration. Benchmark evaluation showed that there was minimal performance gain after 10 trials.


At this stage, it is very important to consider whether you would like anchor selection to be cell type aware or cell type agnostic. If you choose cell type aware, anchors will be restricted between cells of the same type. This approach is optimal if you have high quality cell annotations. If the cell type annotations are poor quality or incomplete, run the model with cell_type_agnostic = True.


In [3]:
study, best_params, adata = analysis.optimize(
    adata,
    n_trials=15,
    cell_type_agnostic=False
)

AttributeError: 'CSHint_integration' object has no attribute 'optimize'

We can find the final embeddings here:

In [ ]:
adata.obsm["X_pca"]

Now, we can check out the umap:

In [ ]:
import scanpy as sc
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata, color= "species")

In [ ]:
sc.pl.umap(adata, color = "cell_type")

Sometimes, you may notice cells that appeared to integrate poorly. These manifest a cells that are not close to their cell-type cluster of origin. This may happen if a cell was not properly labeled or if it's a doublet. These cells can be easily filtered out with the following command:


In [ ]:
adata_pcacor = analysis.post_integration_correction(adata, ["cell_type", "species"], method = "pca", pval_cutoff = pow(10, -10))

Now we can see that they're gone!

In [ ]:
sc.tl.umap(adata_pcacor)
sc.pl.umap(adata_pcacor, color = "cell_type")

Thats it! Now we can save out the embeddings. They are stored in adata.obsm["X_pca"]. In any downstream analysis that you may do, you can sub the merged CSHint embeddings for PCA embeddings.

In [ ]:
import pandas as pd
pca_df = pd.DataFrame(
    adata_pcacor.obsm["X_pca"],
    index=adata_pcacor.obs_names  # cell names
)
pca_df.to_csv("embeddings.csv")

adata_pcacor.obs.to_csv("metadata.csv")

adata_pcacor.write_h5ad("adata.h5ad")